# Medan Magnet dan Hukum Biot-Savart
**Mata Kuliah:** Elektrodinamika Klasik (Program Magister Fisika, Universitas Padjadjaran)  
**Topik:** Magnetostatika

---

## 1. Pendahuluan: Kerangka Teori Magnetostatika

Magnetostatika mengkaji medan magnet $\mathbf{B}$ yang bersifat konstan terhadap waktu, bersumber dari **arus listrik stasioner** (*steady currents*):
$$\frac{\partial \rho}{\partial t} = 0 \implies \nabla \cdot \mathbf{J} = 0$$

Medan magnet diferensial $d\mathbf{B}$ di titik observasi $\mathbf{r}$ akibat elemen arus filamen $I d\mathbf{l}'$ di posisi $\mathbf{r}'$ dinyatakan oleh **Hukum Biot-Savart**:
$$d\mathbf{B}(\mathbf{r}) = \frac{\mu_0 I}{4\pi} \frac{d\mathbf{l}' \times (\mathbf{r} - \mathbf{r}')}{|\mathbf{r} - \mathbf{r}'|^3}$$

Secara universal, medan magnet memenuhi dua persamaan Maxwell magnetostatika:
1. **$\nabla \cdot \mathbf{B} = 0$** (ketiadaan monopol magnetik $\implies \mathbf{B} = \nabla \times \mathbf{A}$).
2. **$\nabla \times \mathbf{B} = \mu_0 \mathbf{J}$** (Hukum Sirkulasi Ampère dalam gauge Coulomb $\nabla \cdot \mathbf{A} = 0$).

Pada notebook ini, **setiap kasus analitik klasik** dikupas secara tuntas melalui tiga pilar komputasi saintifik terpadu:
1. **Penurunan Matematis Analitik & Verifikasi Simbolik (SymPy)**.
2. **Diagram Proyeksi Geometris & Penampang Vektor 2D** yang dirender sebagai geometri Three.js.
3. **Visualisasi 3 Dimensi interaktif Three.js** dengan kontrol kamera, material PBR, konduktor tembaga, dan vektor medan magnet.

In [38]:
# Verifikasi ketersediaan pustaka ilmiah
import sys, subprocess
try:
    import numpy, matplotlib, sympy
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet',
                           'numpy', 'matplotlib', 'sympy'])

In [39]:
import numpy as np
import matplotlib.pyplot as plt
import sympy as sp
import json
import uuid
from IPython.display import HTML, display

# Konstanta Fundamental: Permeabilitas Ruang Hampa
MU_0 = 4.0 * np.pi * 1e-7  # H/m atau T.m/A

# Pengaturan visualisasi matplotlib untuk plot grafik fungsi analitik
plt.rcParams['font.size'] = 11
plt.rcParams['figure.autolayout'] = True

def _three_tube(points, color, radius=0.01, closed=False, metalness=0.0, roughness=0.65, opacity=1.0):
    return {
        'type': 'tube', 'points': np.asarray(points, dtype=float).tolist(),
        'color': color, 'radius': radius, 'closed': closed,
        'metalness': metalness, 'roughness': roughness, 'opacity': opacity
    }

def _three_sphere(center, color, radius=0.05, metalness=0.0, roughness=0.55):
    return {
        'type': 'sphere', 'center': np.asarray(center, dtype=float).tolist(),
        'color': color, 'radius': radius, 'metalness': metalness, 'roughness': roughness
    }

def _three_arrow(start, end, color, radius=0.025, metalness=0.0, roughness=0.4):
    return {
        'type': 'arrow', 'start': np.asarray(start, dtype=float).tolist(),
        'end': np.asarray(end, dtype=float).tolist(), 'color': color,
        'radius': radius, 'metalness': metalness, 'roughness': roughness
    }

def _three_cylinder(center, radius, length, color, opacity=1.0):
    return {
        'type': 'cylinder', 'center': np.asarray(center, dtype=float).tolist(),
        'radius': radius, 'length': length, 'color': color, 'opacity': opacity
    }

def _three_torus(center, major_radius, minor_radius, color, opacity=1.0):
    return {
        'type': 'torus', 'center': np.asarray(center, dtype=float).tolist(),
        'majorRadius': major_radius, 'minorRadius': minor_radius,
        'color': color, 'opacity': opacity
    }

def _three_floor(z, width, depth):
    return {'type': 'floor', 'z': z, 'width': width, 'depth': depth, 'color': '#252530'}

def _three_show(title, elements, camera, target=(0, 0, 0), projection='perspective',
                view_size=4.0, legend=(), height=560):
    container_id = 'three-' + uuid.uuid4().hex
    spec = json.dumps({
        'title': title, 'elements': elements, 'camera': camera, 'target': target,
        'projection': projection, 'viewSize': view_size, 'legend': list(legend)
    }, ensure_ascii=True)
    html = r"""
<div id="__ID__" style="position:relative;width:100%;height:__HEIGHT__px;background:#181820;border-radius:8px;overflow:hidden">
  <div class="three-title" style="position:absolute;z-index:2;left:12px;top:10px;color:#f8fafc;font:600 14px sans-serif;pointer-events:none"></div>
  <button class="three-export-btn" title="Simpan gambar resolusi penuh (PNG)" style="position:absolute;z-index:2;left:12px;bottom:10px;padding:6px 12px;background:#2563eb;color:#ffffff;border:none;border-radius:6px;cursor:pointer;font:500 12px sans-serif;display:flex;align-items:center;gap:6px;box-shadow:0 2px 4px rgba(0,0,0,0.3);transition:background 0.2s;" onmouseover="this.style.background='#1d4ed8'" onmouseout="this.style.background='#2563eb'">
    <svg width="13" height="13" viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="2.2" stroke-linecap="round" stroke-linejoin="round"><path d="M23 19a2 2 0 0 1-2 2H3a2 2 0 0 1-2-2V8a2 2 0 0 1 2-2h4l2-3h6l2 3h4a2 2 0 0 1 2 2z"/><circle cx="12" cy="13" r="4"/></svg>
    Ekspor PNG
  </button>
  <div class="three-legend" style="position:absolute;z-index:2;right:10px;top:42px;max-width:240px;padding:8px 10px;background:#131720dd;border-radius:6px;color:#f8fafc;font:12px sans-serif;pointer-events:none"></div>
  <div class="three-error" style="display:none;position:absolute;z-index:3;left:12px;bottom:46px;color:#fecaca;font:12px sans-serif"></div>
</div>
<script type="importmap">{"imports":{"three":"https://cdn.jsdelivr.net/npm/three@0.169.0/build/three.module.js"}}</script>
<script type="module">
try {
  const THREE = await import('https://cdn.jsdelivr.net/npm/three@0.169.0/build/three.module.js');
  const { OrbitControls } = await import('https://cdn.jsdelivr.net/npm/three@0.169.0/examples/jsm/controls/OrbitControls.js');
  const root = document.getElementById('__ID__');
  const spec = __SPEC__;
  root.querySelector('.three-title').textContent = spec.title;
  const legend = root.querySelector('.three-legend');
  legend.replaceChildren(...spec.legend.map((item) => {
    const row = document.createElement('div');
    row.style.cssText = 'display:flex;align-items:center;gap:7px;margin:3px 0';
    const swatch = document.createElement('span');
    swatch.style.cssText = 'width:10px;height:10px;flex:none;border-radius:50%;background:' + item.color;
    const label = document.createElement('span');
    label.textContent = item.label;
    row.append(swatch, label);
    return row;
  }));
  if (!spec.legend.length) legend.remove();

  const scene = new THREE.Scene();
  scene.background = new THREE.Color('#181820');
  scene.add(new THREE.HemisphereLight(0xe2e8f0, 0x30303a, 2.1));
  const keyLight = new THREE.DirectionalLight(0xffffff, 3.2);
  keyLight.position.set(4, -5, 7);
  keyLight.castShadow = true;
  keyLight.shadow.mapSize.set(1024, 1024);
  scene.add(keyLight);
  const fillLight = new THREE.DirectionalLight(0x93c5fd, 1.1);
  fillLight.position.set(-4, 2, 3);
  scene.add(fillLight);

  const material = (item) => new THREE.MeshStandardMaterial({
    color: item.color, metalness: item.metalness ?? 0.05,
    roughness: item.roughness ?? 0.55, transparent: (item.opacity ?? 1) < 1,
    opacity: item.opacity ?? 1, depthWrite: (item.opacity ?? 1) >= 1,
    side: THREE.DoubleSide
  });
  for (const item of spec.elements) {
    let object;
    if (item.type === 'tube') {
      const points = item.points.map((p) => new THREE.Vector3(...p));
      const curve = new THREE.CatmullRomCurve3(points, item.closed, 'centripetal');
      const geometry = new THREE.TubeGeometry(curve, Math.max(48, points.length * 2), item.radius, 10, item.closed);
      object = new THREE.Mesh(geometry, material(item));
    } else if (item.type === 'sphere') {
      object = new THREE.Mesh(new THREE.SphereGeometry(item.radius, 20, 14), material(item));
      object.position.set(...item.center);
    } else if (item.type === 'arrow') {
      const start = new THREE.Vector3(...item.start);
      const delta = new THREE.Vector3(...item.end).sub(start);
      const length = delta.length();
      if (length < 1e-8) continue;
      const headLength = Math.min(0.17, length * 0.32);
      const headWidth = Math.max(item.radius * 2, headLength * 0.55);
      object = new THREE.ArrowHelper(delta.normalize(), start, length, item.color, headLength, headWidth);
      object.line.material = new THREE.LineBasicMaterial({ color: item.color });
      object.cone.material = material(item);
    } else if (item.type === 'cylinder') {
      object = new THREE.Mesh(
        new THREE.CylinderGeometry(item.radius, item.radius, item.length, 48, 1, true),
        material({ ...item, metalness: 0.05, roughness: 0.75 })
      );
      object.rotation.x = Math.PI / 2;
      object.position.set(...item.center);
    } else if (item.type === 'torus') {
      object = new THREE.Mesh(
        new THREE.TorusGeometry(item.majorRadius, item.minorRadius, 12, 96),
        material({ ...item, metalness: 0.05, roughness: 0.5 })
      );
      object.position.set(...item.center);
    } else if (item.type === 'floor') {
      object = new THREE.Mesh(
        new THREE.PlaneGeometry(item.width, item.depth),
        material({ ...item, metalness: 0.1, roughness: 0.9 })
      );
      object.position.z = item.z;
      object.receiveShadow = true;
    }
    if (object) {
      object.castShadow = true;
      object.receiveShadow = true;
      scene.add(object);
    }
  }

  const width = root.clientWidth || 800;
  const height = root.clientHeight || 560;
  const aspect = width / height;
  let camera;
  if (spec.projection === 'orthographic') {
    const halfHeight = spec.viewSize / 2;
    camera = new THREE.OrthographicCamera(-halfHeight * aspect, halfHeight * aspect, halfHeight, -halfHeight, 0.01, 1000);
  } else {
    camera = new THREE.PerspectiveCamera(42, aspect, 0.01, 1000);
  }
  const viewDirection = new THREE.Vector3(...spec.camera).sub(new THREE.Vector3(...spec.target));
  if (Math.abs(viewDirection.z) > Math.hypot(viewDirection.x, viewDirection.y) * 2) camera.up.set(0, 1, 0);
  else camera.up.set(0, 0, 1);
  camera.position.set(...spec.camera);
  camera.lookAt(...spec.target);
  const renderer = new THREE.WebGLRenderer({ antialias: true, alpha: false, preserveDrawingBuffer: true });
  renderer.setPixelRatio(Math.min(window.devicePixelRatio || 1, 2));
  renderer.setSize(width, height);
  renderer.shadowMap.enabled = true;
  renderer.shadowMap.type = THREE.PCFSoftShadowMap;
  renderer.outputColorSpace = THREE.SRGBColorSpace;
  renderer.toneMapping = THREE.ACESFilmicToneMapping;
  renderer.toneMappingExposure = 1.1;
  root.appendChild(renderer.domElement);
  const controls = new OrbitControls(camera, renderer.domElement);
  controls.target.set(...spec.target);
  controls.enableDamping = false;
  const render = () => renderer.render(scene, camera);
  controls.addEventListener('change', render);
  const resize = new ResizeObserver(() => {
    const w = root.clientWidth, h = root.clientHeight;
    if (!w || !h) return;
    renderer.setSize(w, h);
    if (camera.isPerspectiveCamera) camera.aspect = w / h;
    else {
      const halfHeight = spec.viewSize / 2;
      camera.left = -halfHeight * w / h; camera.right = halfHeight * w / h;
      camera.top = halfHeight; camera.bottom = -halfHeight;
    }
    camera.updateProjectionMatrix();
    render();
  });
  resize.observe(root);
  render();

  const exportBtn = root.querySelector('.three-export-btn');
  if (exportBtn) {
    exportBtn.addEventListener('click', () => {
      render();
      const safeTitle = (spec.title || 'visualisasi_magnetostatik')
        .toLowerCase()
        .replace(/[^a-z0-9]+/g, '_')
        .replace(/^_+|_+$/g, '');
      const link = document.createElement('a');
      link.download = (safeTitle || 'visualisasi') + '.png';
      link.href = renderer.domElement.toDataURL('image/png');
      link.click();
    });
  }
} catch (error) {
  const root = document.getElementById('__ID__');
  const message = root.querySelector('.three-error');
  message.textContent = 'Three.js gagal dimuat: ' + error.message + '. Periksa koneksi ke CDN.';
  message.style.display = 'block';
  console.error(error);
}
</script>
"""
    html = (html.replace('__ID__', container_id)
                .replace('__HEIGHT__', str(height))
                .replace('__SPEC__', spec))
    display(HTML(html))

print(f"Modul berhasil dimuat: NumPy, Matplotlib, SymPy, Three.js. (mu_0 = {MU_0:.6e} T.m/A)")

Modul berhasil dimuat: NumPy, Matplotlib, SymPy, Three.js. (mu_0 = 1.256637e-06 T.m/A)


---
## 2. Kasus Analitik 1: Kawat Lurus Berarus (Panjang Berhingga & Tak Berhingga)

### 2.1 Penurunan Matematis Eksak
Tinjau sebuah kawat lurus tipis yang terletak pada sumbu-$z$ dari $z' = z_1$ hingga $z' = z_2$, dialiri arus konstan $I$ ke arah $+z$. Titik observasi berada pada jarak radial tegak lurus $s$ dari kawat pada koordinat silinder $(s, \phi, z = 0)$.

- Vektor posisi sumber: $\mathbf{r}' = z' \hat{\mathbf{z}} \implies d\mathbf{l}' = dz' \hat{\mathbf{z}}$
- Vektor posisi titik observasi: $\mathbf{r} = s \hat{\mathbf{s}}$
- Vektor perpindahan: $\boldsymbol{r} = \mathbf{r} - \mathbf{r}' = s \hat{\mathbf{s}} - z' \hat{\mathbf{z}}$
- Magnitudo jarak: $|\boldsymbol{r}| = \sqrt{s^2 + z'^2}$

Perkalian silang:
$$d\mathbf{l}' \times (\mathbf{r} - \mathbf{r}') = (dz' \hat{\mathbf{z}}) \times (s \hat{\mathbf{s}} - z' \hat{\mathbf{z}}) = s dz' (\hat{\mathbf{z}} \times \hat{\mathbf{s}}) = s dz' \hat{\boldsymbol{\phi}}$$

Maka vektor medan diferensial murni berarah azimutal melingkar:
$$d\mathbf{B} = \frac{\mu_0 I s}{4\pi (s^2 + z'^2)^{3/2}} dz' \, \hat{\boldsymbol{\phi}}$$

Gunakan substitusi trigonometri sudut polar $\theta$ (sudut antara garis perpindahan dengan sumbu-$z$):
$$\tan\theta = \frac{s}{z'} \implies z' = s \cot\theta \implies dz' = -s \csc^2\theta \, d\theta$$
$$s^2 + z'^2 = s^2 \csc^2\theta$$

Integrasi dari sudut $\theta_1$ (ujung bawah) hingga $\theta_2$ (ujung atas):
$$B_\phi = \frac{\mu_0 I s}{4\pi} \int_{\theta_1}^{\theta_2} \frac{-s \csc^2\theta \, d\theta}{s^3 \csc^3\theta} = -\frac{\mu_0 I}{4\pi s} \int_{\theta_1}^{\theta_2} \sin\theta \, d\theta = \frac{\mu_0 I}{4\pi s} (\cos\theta_2 - \cos\theta_1)$$

Jika diukur dengan sudut elevasi $\alpha_1$ dan $\alpha_2$ dari bidang ekuator ($z=0$):
$$B_\phi = \frac{\mu_0 I}{4\pi s} (\sin\alpha_2 - \sin\alpha_1)$$

Untuk kawat lurus **tak berhingga** ($z_1 \to -\infty, z_2 \to +\infty \implies \alpha_1 = -\pi/2, \alpha_2 = +\pi/2$):
$$B_\phi(s) = \frac{\mu_0 I}{2\pi s}$$

### 2.2 Penjelasan Fisis & Proyeksi Vektor
Secara fisis, perkalian silang $d\mathbf{l}' \times \hat{\boldsymbol{r}}$ selalu berada pada bidang yang tegak lurus terhadap bidang yang dibentuk oleh kawat dan garis jarak $s$. Oleh karena itu:
- Komponen aksial $B_z$ dan radial $B_s$ **lenyap murni secara matematis** untuk setiap elemen kawat tunggal.
- Garis gaya medan membentuk lingkaran-lingkaran tertutup konsentris mengitari kawat searah kaidah tangan kanan putaran sekrup.

In [40]:
# Verifikasi Simbolik SymPy: Kawat Lurus Tak Hingga
mu0, I, s, z_prime = sp.symbols('mu_0 I s z_prime', positive=True)
integrand_wire = s / (s**2 + z_prime**2)**sp.Rational(3, 2)
def_wire = sp.integrate(integrand_wire, (z_prime, -sp.oo, sp.oo))
B_wire_sym = (mu0 * I / (4 * sp.pi)) * def_wire
print("Hasil Verifikasi SymPy (Kawat Lurus Tak Hingga):")
sp.pprint(sp.simplify(B_wire_sym))

Hasil Verifikasi SymPy (Kawat Lurus Tak Hingga):
I⋅μ₀ 
─────
2⋅π⋅s


In [41]:
# Diagram Proyeksi Vektor 2D: Geometri Kawat Lurus Berarus
s_obs = 2.0
elements = [
    _three_tube([[0, 0, -2], [0, 0, 2]], '#d97443', radius=0.04, metalness=0.8, roughness=0.2),
    _three_arrow([0, 0, 0.85], [0, 0, 1.35], '#ff3333', radius=0.04),
    _three_sphere([0, 0, 0.8], '#ff3333', radius=0.07),
    _three_arrow([s_obs, 0, -0.1], [s_obs, 0, 0.25], '#38bdf8', radius=0.035),
    _three_tube([[0, 0, 0.8], [s_obs, 0, 0]], '#c084fc', radius=0.015),
    _three_tube([[0, 0, 0], [s_obs, 0, 0]], '#94a3b8', radius=0.01),
    _three_arrow([s_obs, 0, 0], [s_obs, 0.42, 0], '#00f5d4', radius=0.035),
]
_three_show(
    'Proyeksi Vektor 2D: Geometri Kawat Lurus Berarus',
    elements, camera=[1.0, -9.0, 0.6], target=[1.0, 0, 0],
    projection='orthographic', view_size=5.2,
    legend=[
        {'color': '#d97443', 'label': 'Kawat berarus I ke +z'},
        {'color': '#ff3333', 'label': "Elemen arus I dz'"},
        {'color': '#38bdf8', 'label': 'Titik observasi'},
        {'color': '#c084fc', 'label': 'Vektor perpindahan r'},
        {'color': '#00f5d4', 'label': 'Medan B azimutal'}
    ], height=480
)

In [42]:
# Visualisasi 3D: Kawat Tembaga dan Garis Medan Azimutal
elements = [
    _three_tube([[0, 0, -1.5], [0, 0, 1.5]], '#d97443', radius=0.035, metalness=0.88, roughness=0.18),
    _three_arrow([0, 0, 0.34], [0, 0, 0.72], '#ff3333', radius=0.04, metalness=0.6),
    _three_floor(-1.5, 3.0, 3.0)
]
radii_w = [0.35, 0.65, 0.95]
colors_w = ['#00f5d4', '#00bbf9', '#f15bb5']
for r_val, color in zip(radii_w, colors_w):
    for z0 in [-0.5, 0.0, 0.5]:
        angles = np.linspace(0, 2*np.pi, 120)
        ring = np.column_stack([r_val*np.cos(angles), r_val*np.sin(angles), np.full_like(angles, z0)])
        elements.append(_three_tube(ring, color, radius=0.012, closed=True, metalness=0.45, roughness=0.25))
        for angle in [0, np.pi/2, np.pi, 3*np.pi/2]:
            start = np.array([r_val*np.cos(angle), r_val*np.sin(angle), z0])
            tangent = np.array([-np.sin(angle), np.cos(angle), 0.0])
            elements.append(_three_arrow(start, start + 0.16*tangent, color, radius=0.025))
_three_show(
    'Kawat Tembaga dan Garis Medan Magnet Azimutal',
    elements, camera=[2.6, -2.6, 1.8], target=[0, 0, 0],
    legend=[
        {'color': '#d97443', 'label': 'Kawat tembaga, arus ke +z'},
        {'color': '#00f5d4', 'label': 'Garis medan B (s = 0.35 m)'},
        {'color': '#00bbf9', 'label': 'Garis medan B (s = 0.65 m)'},
        {'color': '#f15bb5', 'label': 'Garis medan B (s = 0.95 m)'}
    ]
)

---
## 3. Kasus Analitik 2: Cincin Melingkar Berarus (Circular Current Loop)

### 3.1 Penurunan Matematis Eksak
Tinjau loop melingkar berjari-jari $R$ pada bidang $xy$ berpusat di titik asal $(0, 0, 0)$ yang dialiri arus $I$ berlawanan arah jarum jam (dilihat dari $+z$).

Titik observasi berada pada sumbu simetri $z$, $\mathbf{r} = z \hat{\mathbf{z}}$.
- Elemen sumber: $\mathbf{r}' = R\cos\phi' \hat{\mathbf{x}} + R\sin\phi' \hat{\mathbf{y}}$
- Vektor diferensial: $d\mathbf{l}' = (-R\sin\phi' \hat{\mathbf{x}} + R\cos\phi' \hat{\mathbf{y}}) d\phi'$
- Vektor perpindahan: $\boldsymbol{r} = \mathbf{r} - \mathbf{r}' = -R\cos\phi' \hat{\mathbf{x}} - R\sin\phi' \hat{\mathbf{y}} + z \hat{\mathbf{z}}$
- Jarak: $|\boldsymbol{r}| = \sqrt{R^2 + z^2}$

Perkalian silang:
$$d\mathbf{l}' \times \boldsymbol{r} = \left( R z \cos\phi' \hat{\mathbf{x}} + R z \sin\phi' \hat{\mathbf{y}} + R^2 \hat{\mathbf{z}} \right) d\phi'$$

Ketika diintegrasikan terhadap $\phi' \in [0, 2\pi]$:
$$\int_0^{2\pi} \cos\phi' \, d\phi' = 0, \quad \int_0^{2\pi} \sin\phi' \, d\phi' = 0$$
Maka komponen medan pada bidang $xy$ ($B_x$ dan $B_y$) lenyap sempurna! Komponen yang tersisa murni aksial:
$$\mathbf{B}(z) = \frac{\mu_0 I}{4\pi (R^2 + z^2)^{3/2}} \int_0^{2\pi} R^2 d\phi' \, \hat{\mathbf{z}} = \frac{\mu_0 I R^2}{2(R^2 + z^2)^{3/2}} \hat{\mathbf{z}}$$

### 3.2 Penjelasan Fisis & Proyeksi Vektor
Untuk setiap elemen arus $d\mathbf{l}'_1$ pada sudut $\phi'$, selalu ada elemen pasangan antipodal $d\mathbf{l}'_2$ pada $\phi' + \pi$.
- Vektor medan $d\mathbf{B}_1$ dan $d\mathbf{B}_2$ membentuk sudut kemiringan terhadap sumbu-$z$.
- Proyeksi radial $dB_\perp$ dari kedua elemen berlawanan arah tepat $180^\circ$ sehingga saling membatalkan.
- Proyeksi aksial $dB_\parallel = dB \cos\alpha$ searah sumbu $+z$ saling menjumlahkan secara konstruktif.

In [43]:
# Verifikasi Simbolik SymPy: Medan Loop Melingkar di Sumbu-Z
R_sym, z_sym, phi_sym = sp.symbols('R z phi_prime', positive=True, real=True)
dl_sym = sp.Matrix([-R_sym * sp.sin(phi_sym), R_sym * sp.cos(phi_sym), 0])
r_sym = sp.Matrix([-R_sym * sp.cos(phi_sym), -R_sym * sp.sin(phi_sym), z_sym])
cross_sym = dl_sym.cross(r_sym)
B_int = cross_sym / (R_sym**2 + z_sym**2)**sp.Rational(3, 2)
B_loop_sym = (mu0 * I / (4 * sp.pi)) * sp.integrate(B_int, (phi_sym, 0, 2*sp.pi))
print("Hasil Verifikasi SymPy (Medan Loop di Sumbu-Z):")
sp.pprint(B_loop_sym)

Hasil Verifikasi SymPy (Medan Loop di Sumbu-Z):
⎡      0       ⎤
⎢              ⎥
⎢      0       ⎥
⎢              ⎥
⎢      2       ⎥
⎢   I⋅R ⋅μ₀    ⎥
⎢──────────────⎥
⎢           3/2⎥
⎢  ⎛ 2    2⎞   ⎥
⎣2⋅⎝R  + z ⎠   ⎦


In [44]:
# Diagram Proyeksi Vektor 2D: Pembatalan Komponen Radial Loop Melingkar
R_l = 1.2
z_o = 1.4
elements = []
elements += [_three_sphere([-R_l, 0, 0], '#ef4444', 0.08), _three_sphere([R_l, 0, 0], '#3b82f6', 0.08)]
angles = np.linspace(0, 2*np.pi, 120)
elements.append(_three_tube(np.column_stack([R_l*np.cos(angles), np.zeros_like(angles), 0.15*np.sin(angles)]), '#64748b', radius=0.015, closed=True))
elements.append(_three_arrow([0, 0, z_o-0.12], [0, 0, z_o+0.15], '#f8fafc', radius=0.035))
elements.append(_three_tube([[-R_l, 0, 0], [0, 0, z_o]], '#ef4444', radius=0.012))
elements.append(_three_tube([[R_l, 0, 0], [0, 0, z_o]], '#3b82f6', radius=0.012))
dB1_d = np.array([0.5, 0, 0.5])
dB1_e = np.array([0, 0, z_o]) + dB1_d
elements.append(_three_arrow([0, 0, z_o], dB1_e, '#f43f5e', radius=0.035))
dB2_d = np.array([-0.5, 0, 0.5])
dB2_e = np.array([0, 0, z_o]) + dB2_d
elements.append(_three_arrow([0, 0, z_o], dB2_e, '#38bdf8', radius=0.035))
dB_s_e = np.array([0, 0, z_o]) + np.array([0, 0, 1.0])
elements.append(_three_arrow([0, 0, z_o], dB_s_e, '#10b981', radius=0.05))
elements.append(_three_tube([[dB1_e[0], 0, dB1_e[2]], [0, 0, dB1_e[2]]], '#f43f5e', radius=0.008))
elements.append(_three_tube([[dB2_e[0], 0, dB2_e[2]], [0, 0, dB2_e[2]]], '#38bdf8', radius=0.008))
_three_show(
    'Proyeksi Vektor 2D: Pembatalan Komponen Radial Loop Melingkar',
    elements, camera=[0, -10, 0.1], target=[0, 0, 0.8],
    projection='orthographic', view_size=4.2,
    legend=[
        {'color': '#ef4444', 'label': 'Elemen arus keluar bidang'},
        {'color': '#3b82f6', 'label': 'Elemen arus masuk bidang'},
        {'color': '#f43f5e', 'label': 'dB₁'},
        {'color': '#38bdf8', 'label': 'dB₂'},
        {'color': '#10b981', 'label': 'Resultan aksial dBz'}
    ], height=480
)

In [45]:
# Visualisasi 3D: Loop Melingkar dan Topologi Medan Dipol
R_loop = 0.5
elements = []
angles_loop = np.linspace(0, 2*np.pi, 150)
pts_loop = np.column_stack([R_loop*np.cos(angles_loop), R_loop*np.sin(angles_loop), np.zeros_like(angles_loop)])
elements.append(_three_tube(pts_loop, '#e07a5f', radius=0.03, closed=True, metalness=0.88, roughness=0.18))
for angle in [np.pi/4, 5*np.pi/4]:
    pos = np.array([R_loop*np.cos(angle), R_loop*np.sin(angle), 0])
    tangent = np.array([-np.sin(angle), np.cos(angle), 0])
    elements.append(_three_arrow(pos - 0.08*tangent, pos + 0.10*tangent, '#ff3333', radius=0.03, metalness=0.6))

# Evaluasi numerik Biot-Savart pada grid 3D
pts_eval_l = []
for z_v in np.linspace(-0.65, 0.65, 7):
    for r_v in [0.0, 0.25, 0.55, 0.85]:
        for a_v in np.linspace(0, 2*np.pi, 8 if r_v > 0 else 1, endpoint=False):
            pts_eval_l.append([r_v*np.cos(a_v), r_v*np.sin(a_v), z_v])
pts_eval_l = np.array(pts_eval_l, dtype=np.float32)
N_disc_l = 50
phi_dl = np.linspace(0, 2*np.pi, N_disc_l, endpoint=False)
nodes_dl = np.column_stack([R_loop*np.cos(phi_dl), R_loop*np.sin(phi_dl), np.zeros(N_disc_l)])
dl_vec = np.column_stack([-R_loop*np.sin(phi_dl)*(2*np.pi/N_disc_l),
                          R_loop*np.cos(phi_dl)*(2*np.pi/N_disc_l), np.zeros(N_disc_l)])
r_disp_l = pts_eval_l[:, np.newaxis, :] - nodes_dl[np.newaxis, :, :]
dist_l = np.linalg.norm(r_disp_l, axis=2)
dist_safe_l = np.where(dist_l < 0.12, np.inf, dist_l)
factor_l = np.cross(dl_vec[np.newaxis, :, :], r_disp_l) / (dist_safe_l[:, :, np.newaxis]**3)
B_eval_l = (MU_0*10.0/(4*np.pi))*np.sum(factor_l, axis=1)
B_mag_l = np.linalg.norm(B_eval_l, axis=1)
mask_valid_l = (B_mag_l > 1e-7) & (B_mag_l < 5e-5)
strength = B_mag_l[mask_valid_l]
for point, vector, magnitude in zip(pts_eval_l[mask_valid_l], B_eval_l[mask_valid_l], strength):
    direction = vector / (magnitude + 1e-12)
    fraction = np.clip((magnitude - strength.min())/(np.ptp(strength) + 1e-12), 0, 1)
    color = plt.cm.plasma(fraction)
    color_hex = '#%02x%02x%02x' % tuple(int(255*c) for c in color[:3])
    elements.append(_three_arrow(point, point + 0.12*direction, color_hex, radius=0.022))
elements.append(_three_floor(-1.0, 3.0, 3.0))
_three_show(
    'Loop Melingkar dan Vektor Medan Biot–Savart',
    elements, camera=[2.0, -2.2, 1.6], target=[0, 0, 0],
    legend=[
        {'color': '#e07a5f', 'label': 'Kawat loop tembaga (I = 10 A)'},
        {'color': '#6a00a8', 'label': 'Vektor B lemah'},
        {'color': '#f0e442', 'label': 'Vektor B kuat'}
    ]
)

---
## 4. Kasus Analitik 3: Solenoida Silinder (Berhingga & Tak Berhingga)

### 4.1 Penurunan Matematis Eksak
Tinjau sebuah solenoida silinder dengan radius $R$, panjang $L$ dari $z_1 = -L/2$ sampai $z_2 = +L/2$, memiliki kerapatan lilitan $n = N/L$ dan dialiri arus $I$.

Setiap irisan cincin setebal $dz'$ memuat arus diferensial $dI = n I dz'$. Berdasarkan hasil Kasus 2, kontribusi medan aksial pada sumbu-$z$ adalah:
$$dB_z = \frac{\mu_0 n I R^2}{2 [R^2 + (z - z')^2]^{3/2}} dz'$$

Dengan substitusi sudut polar $\tan\theta = \frac{R}{z' - z} \implies z' - z = R \cot\theta$:
$$dz' = -R \csc^2\theta \, d\theta$$
$$B_z(z) = \frac{\mu_0 n I}{2} \int_{\theta_1}^{\theta_2} (-\sin\theta) \, d\theta = \frac{\mu_0 n I}{2} (\cos\theta_2 - \cos\theta_1)$$

Dalam bentuk koordinat posisi:
$$B_z(z) = \frac{\mu_0 n I}{2} \left[ \frac{z_2 - z}{\sqrt{R^2 + (z_2 - z)^2}} - \frac{z_1 - z}{\sqrt{R^2 + (z_1 - z)^2}} \right]$$

- **Pusat Solenoida Tak Hingga ($L \to \infty$):** $B_z = \mu_0 n I$ (homogen sempurna).
- **Tepi Ujung Solenoida:** $B_z = \frac{1}{2}\mu_0 n I$.

### 4.2 Penjelasan Fisis & Proyeksi Vektor
Di dalam rongga internal solenoida, garis-garis gaya dari seluruh lilitan sejajar dan saling memperkuat ke arah aksial $+z$. Di luar solenoida, garis gaya berbelok kembali membentuk loop tertutup, menyebar ke ruang yang sangat luas sehingga intensitas medan luar mendekati nol saat $L \to \infty$.

In [46]:
# Verifikasi Simbolik SymPy: Solenoida Berhingga & Limit Tak Hingga
z_obs, zp, L_s, R_s, n_s, I_s = sp.symbols('z z_prime L R n I', positive=True, real=True)
integrand_s = (mu0 * n_s * I_s * R_s**2) / (2 * (R_s**2 + (z_obs - zp)**2)**sp.Rational(3, 2))
B_sol_expr = sp.integrate(integrand_s, (zp, -L_s/2, L_s/2))
B_sol_center = B_sol_expr.subs(z_obs, 0)
B_sol_inf_sym = sp.limit(B_sol_center, L_s, sp.oo)
print("Hasil Verifikasi SymPy (Medan di Pusat Solenoida L -> oo):")
sp.pprint(B_sol_inf_sym)

Hasil Verifikasi SymPy (Medan di Pusat Solenoida L -> oo):
I⋅μ₀⋅n


In [47]:
# Diagram Proyeksi Vektor 2D: Geometri Longitudinal Solenoida
z_wires = np.linspace(-1.2, 1.2, 13)
R_sol = 0.6
elements = []
for zw in z_wires:
    elements.append(_three_sphere([zw, 0, R_sol], '#ef4444', 0.045))
    elements.append(_three_sphere([zw, 0, -R_sol], '#3b82f6', 0.045))
elements.append(_three_tube([[-1.8, 0, 0], [1.8, 0, 0]], '#94a3b8', radius=0.008))
zp = 0.2
elements.append(_three_arrow([zp, 0, -0.08], [zp, 0, 0.12], '#f8fafc', radius=0.03))
elements.append(_three_tube([[-1.2, 0, R_sol], [zp, 0, 0]], '#10b981', radius=0.012))
elements.append(_three_tube([[1.2, 0, R_sol], [zp, 0, 0]], '#a855f7', radius=0.012))
for xi in np.linspace(-0.8, 0.6, 5):
    elements.append(_three_arrow([xi, 0, 0], [xi + 0.25, 0, 0], '#00f5d4', radius=0.035))
_three_show(
    'Proyeksi Vektor 2D: Irisan Longitudinal Solenoida',
    elements, camera=[0, -10, 0.1], target=[0, 0, 0],
    projection='orthographic', view_size=4.2,
    legend=[
        {'color': '#ef4444', 'label': 'Arus dinding atas (keluar)'},
        {'color': '#3b82f6', 'label': 'Arus dinding bawah (masuk)'},
        {'color': '#94a3b8', 'label': 'Sumbu simetri aksial'},
        {'color': '#00f5d4', 'label': 'Medan Bz homogen di inti'}
    ], height=480
)

In [48]:
# Visualisasi 3D: Heliks Solenoida dan Medan Homogen Rongga
R_hel = 0.35
L_hel = 1.2
N_hel = 10
t_hel = np.linspace(0, 2*np.pi*N_hel, 800)
pts_hel = np.column_stack([R_hel*np.cos(t_hel), R_hel*np.sin(t_hel), np.linspace(-L_hel/2, L_hel/2, 800)])
elements = [
    _three_tube(pts_hel, '#f4a261', radius=0.02, metalness=0.9, roughness=0.16),
    _three_cylinder([0, 0, 0], R_hel*1.02, L_hel, '#8ecae6', opacity=0.14)
]
z_in = np.linspace(-0.45, 0.45, 5)
r_in = [0.0, 0.15, 0.25]
for zv in z_in:
    for rv in r_in:
        angles = np.linspace(0, 2*np.pi, 6 if rv > 0 else 1, endpoint=False)
        for angle in angles:
            point = np.array([rv*np.cos(angle), rv*np.sin(angle), zv])
            elements.append(_three_arrow(point, point + [0, 0, 0.14], '#00f5d4', radius=0.023, metalness=0.4))
for rv in [0.08, 0.20]:
    for angle in np.linspace(0, 2*np.pi, 4, endpoint=False):
        flux = np.column_stack([
            np.full(50, rv*np.cos(angle)), np.full(50, rv*np.sin(angle)),
            np.linspace(-L_hel/2*0.9, L_hel/2*0.9, 50)
        ])
        elements.append(_three_tube(flux, '#00bbf9', radius=0.006))
elements.append(_three_floor(-1.0, 3.0, 3.0))
_three_show(
    'Kumparan Heliks Solenoida dan Medan Homogen',
    elements, camera=[2.4, -2.4, 1.5], target=[0, 0, 0],
    legend=[
        {'color': '#f4a261', 'label': f'Lilitan solenoida ({N_hel} turns)'},
        {'color': '#8ecae6', 'label': 'Selongsong transparan'},
        {'color': '#00f5d4', 'label': 'Vektor medan aksial internal'},
        {'color': '#00bbf9', 'label': 'Garis fluks medan'}
    ]
)

---
## 5. Kasus Analitik 4: Toroida (Toroidal Solenoid) — Analisis Matematis Komprehensif & Proyeksi Vektor

Sebagaimana solenoida pada Kasus 3 merupakan superposisi kontinu dari loop-loop melingkar linear, **toroida** secara geometris terbentuk ketika solenoida silinder sepanjang $L = 2\pi R$ ditekuk melingkar ujung ke ujung membentuk cincin tertutup (*doughnut / torus*).

### 5.1 Parameter Geometri Toroida
Tinjau sebuah toroida dengan spesifikasi:
- Jari-jari dalam: $a$
- Jari-jari luar: $b$
- Jari-jari rata-rata: $R = \frac{a+b}{2}$
- Lebar penampang radial: $w = b - a$
- Tinggi penampang vertikal: $h$ (untuk penampang persegi panjang) atau radius penampang $r_0 = \frac{b-a}{2}$ (untuk penampang lingkaran)
- Jumlah lilitan total: $N$
- Kuat arus listrik konstan: $I$

Gunakan sistem koordinat silinder $(s, \phi, z)$ di mana sumbu-$z$ berimpit dengan sumbu simetri rotasi utama toroida.

---

### 5.2 Analisis Proyeksi Vektor & Pembuktian Teorema Simetri Biot-Savart
Mari kita bedah secara matematis dan fisis mengapa medan magnet toroida ideal **murni memiliki komponen azimutal $B_\phi$**, sedangkan komponen aksial $B_z$ dan radial $B_s$ lenyap secara eksak di seluruh ruang:

#### 1. Pembuktian Pembatalan Komponen Aksial ($B_z = 0$):
Tinjau penampang melintang lilitan kawat. Aliran arus dapat diuraikan menjadi empat segmen:
- **Segmen dalam ($s' = a$):** Arus mengalir vertikal ke atas: $d\mathbf{l}' = dz' \hat{\mathbf{z}}$.
- **Segmen luar ($s' = b$):** Arus mengalir vertikal ke bawah: $d\mathbf{l}' = -dz' \hat{\mathbf{z}}$.
- **Segmen atas ($z' = +h/2$):** Arus mengalir radial ke luar: $d\mathbf{l}' = ds' \hat{\mathbf{s}}'$.
- **Segmen bawah ($z' = -h/2$):** Arus mengalir radial ke dalam: $d\mathbf{l}' = -ds' \hat{\mathbf{s}}'$.

Untuk menghitung komponen $B_z$ pada sembarang titik observasi $\mathbf{r} = (s, \phi, z)$, perkalian silang Biot-Savart $d\mathbf{l}' \times (\mathbf{r} - \mathbf{r}')$ pada segmen atas dan bawah menghasilkan:
$$[ds' \hat{\mathbf{s}}' \times (\mathbf{r} - \mathbf{r}')]_z \propto \sin(\phi - \phi')$$
Ketika diintegrasikan terhadap sudut azimutal lilitan $\phi' \in [0, 2\pi]$ yang terdistribusi seragam ($\frac{dN}{d\phi'} = \frac{N}{2\pi}$):
$$\int_0^{2\pi} \sin(\phi - \phi') \, d\phi' = 0 \implies B_z = 0$$
Artinya, komponen aksial vertikal dari seluruh lilitan **saling membatalkan secara simetri rotasi penuh**.

#### 2. Pembuktian Pembatalan Komponen Radial ($B_s = 0$):
Berdasarkan hukum Gauss untuk magnetisme:
$$\nabla \cdot \mathbf{B} = 0 \implies \frac{1}{s} \frac{\partial}{\partial s}(s B_s) + \frac{1}{s} \frac{\partial B_\phi}{\partial \phi} + \frac{\partial B_z}{\partial z} = 0$$
Karena simetri rotasi aksial menuntut invariansi terhadap $\phi$ ($\frac{\partial B_\phi}{\partial \phi} = 0$) dan kita telah membuktikan $B_z = 0$, maka:
$$\frac{\partial}{\partial s}(s B_s) = 0 \implies s B_s = \text{konstanta} \implies B_s(s) = \frac{C}{s}$$
Jika konstanta $C \neq 0$, maka akan terdapat fluks magnetik radial netto yang menembus permukaan silinder tertutup di sekitar sumbu-$z$, yang menuntut adanya kerapatan monopol magnetik garis di sumbu-$z$. Karena monopol magnetik tidak ada secara fisik, maka konstanta $C = 0$, sehingga:
$$B_s = 0$$

#### 3. Keberadaan Komponen Azimutal ($B_\phi \neq 0$):
Arus vertikal ke atas pada dinding dalam ($s'=a$) dan ke bawah pada dinding luar ($s'=b$) menghasilkan vektor perkalian silang $d\mathbf{l}' \times \boldsymbol{r}$ yang berproyeksi positif searah lintasan lingkaran azimutal $\hat{\boldsymbol{\phi}}$ (aturan tangan kanan). Kontribusi dari seluruh $N$ lilitan berinterferensi secara konstruktif mengelilingi sumbu-$z$.

---

### 5.3 Penurunan Eksak Melalui Hukum Sirkulasi Ampère
Karena simetri rotasi memastikan $\mathbf{B}(\mathbf{r}) = B_\phi(s) \hat{\boldsymbol{\phi}}$, kita pilih loop Amperian berupa lingkaran horizontal konsentris berjari-jari $s$ pada bidang $z = 0$:

$$\oint_C \mathbf{B} \cdot d\mathbf{l} = \int_0^{2\pi} B_\phi(s) (s \, d\phi) = 2\pi s \, B_\phi(s)$$

Menurut Hukum Ampère:
$$\oint_C \mathbf{B} \cdot d\mathbf{l} = \mu_0 I_{\text{enc}}$$

Kita evaluasi arus yang dilingkupi ($I_{\text{enc}}$) pada tiga wilayah spasial:

1. **Wilayah I: Rongga Silinder Dalam ($s < a$)**  
   Permukaan Amperian berada di lubang pusat toroida. Tidak ada kawat berarus yang tertembus oleh permukaan ini:
   $$I_{\text{enc}} = 0 \implies 2\pi s B_\phi(s) = 0 \implies B_\phi(s) = 0$$

2. **Wilayah II: Di Dalam Inti Toroida ($a \le s \le b$)**  
   Permukaan Amperian memotong seluruh $N$ kawat dinding dalam yang masing-masing membawa arus $I$ ke atas:
   $$I_{\text{enc}} = N I$$
   $$2\pi s B_\phi(s) = \mu_0 N I \implies B_\phi(s) = \frac{\mu_0 N I}{2\pi s}$$
   > **Sifat Fisika Fundamental:** Kuat medan magnet di dalam toroida **tidak seragam secara radial**. Kuat medan berbanding terbalik dengan jarak radial $s$ ($B \propto 1/s$). Kuat medan mencapai maksimum di dinding dalam $s = a$ dan minimum di dinding luar $s = b$:
   > $$B_{\max} = \frac{\mu_0 N I}{2\pi a}, \quad B_{\min} = \frac{\mu_0 N I}{2\pi b}$$

3. **Wilayah III: Di Luar Toroida ($s > b$)**  
   Permukaan Amperian melingkupi seluruh toroida, sehingga memotong $N$ kawat dinding dalam (arus $+I$) dan $N$ kawat dinding luar (arus $-I$):
   $$I_{\text{enc}} = N I - N I = 0 \implies B_\phi(s) = 0$$

#### Kaitan Asimtotik dengan Solenoida Ideal (Limit $R \gg w$):
Jika jari-jari rata-rata toroida $R$ jauh lebih besar daripada ketebalan radialnya ($w = b - a \ll R$), maka untuk seluruh titik di dalam inti $s \approx R$:
$$B_\phi \approx \frac{\mu_0 N I}{2\pi R} = \mu_0 \left(\frac{N}{2\pi R}\right) I = \mu_0 n I$$
di mana $n = \frac{N}{2\pi R}$ adalah kerapatan lilitan per satuan keliling toroida. Nilai ini **persis identik dengan medan solenoida ideal tak terhingga** pada Kasus 3!

In [49]:
# Verifikasi Simbolik SymPy: Fluks Magnetik & Induktansi Diri Toroida
mu0_s, N_s, I_s, a_s, b_s, h_s, s_var = sp.symbols('mu_0 N I a b h s', positive=True, real=True)

# Kuat medan magnetik di dalam inti toroida B(s)
B_tor_sym = (mu0_s * N_s * I_s) / (2 * sp.pi * s_var)

# Elemen luas penampang persegi panjang: da = h * ds
da = h_s

# Fluks magnetik total yang menembus satu lilitan penampang toroida: Phi = integral B da
Phi_m = sp.integrate(B_tor_sym * da, (s_var, a_s, b_s))

# Induktansi Diri Toroida: L = (N * Phi) / I
L_tor_sym = sp.simplify((N_s * Phi_m) / I_s)

print("Formula Eksak Fluks Magnetik Satu Lilitan Toroida Persegi Panjang:")
sp.pprint(Phi_m)

print("\nFormula Induktansi Diri Toroida (L = N*Phi / I):")
sp.pprint(L_tor_sym)

# Limit toroida tipis: b = a + w dengan w << a
w_s = sp.symbols('w', positive=True)
L_approx = L_tor_sym.subs(b_s, a_s + w_s)
L_series = sp.series(L_approx, w_s, 0, 2).removeO()
print("\nLimit Toroida Tipis (w << a):")
sp.pprint(sp.simplify(L_series))

Formula Eksak Fluks Magnetik Satu Lilitan Toroida Persegi Panjang:
  I⋅N⋅h⋅μ₀⋅log(a)   I⋅N⋅h⋅μ₀⋅log(b)
- ─────────────── + ───────────────
        2⋅π               2⋅π      

Formula Induktansi Diri Toroida (L = N*Phi / I):
   ⎛    2     ⎞
   ⎜   N ⋅h⋅μ₀⎟
   ⎜   ───────⎟
   ⎜     2⋅π  ⎟
   ⎜⎛b⎞       ⎟
log⎜⎜─⎟       ⎟
   ⎝⎝a⎠       ⎠

Limit Toroida Tipis (w << a):
 2       
N ⋅h⋅μ₀⋅w
─────────
  2⋅π⋅a  


In [50]:
# Diagram Proyeksi Vektor 2D: Penampang Horizontal Toroida
a_t, b_t = 0.7, 1.3
angles = np.linspace(0, 2*np.pi, 120)
c_in = np.column_stack([a_t*np.cos(angles), a_t*np.sin(angles), np.zeros_like(angles)])
c_out = np.column_stack([b_t*np.cos(angles), b_t*np.sin(angles), np.zeros_like(angles)])
c_amp = np.column_stack([np.cos(angles), np.sin(angles), np.zeros_like(angles)])
elements = [
    _three_tube(c_in, '#cbd5e1', radius=0.012, closed=True),
    _three_tube(c_out, '#cbd5e1', radius=0.012, closed=True),
    _three_tube(c_amp, '#f43f5e', radius=0.015, closed=True)
]
for angle in np.linspace(0, 2*np.pi, 16, endpoint=False):
    elements.append(_three_sphere([a_t*np.cos(angle), a_t*np.sin(angle), 0], '#ef4444', 0.04))
    elements.append(_three_sphere([b_t*np.cos(angle), b_t*np.sin(angle), 0], '#3b82f6', 0.04))
for angle in np.linspace(0, 2*np.pi, 8, endpoint=False):
    pos = np.array([np.cos(angle), np.sin(angle), 0])
    tangent = np.array([-np.sin(angle), np.cos(angle), 0])
    elements.append(_three_arrow(pos, pos + 0.22*tangent, '#00f5d4', radius=0.03))
_three_show(
    'Proyeksi Vektor 2D: Penampang Horizontal Toroida',
    elements, camera=[0, 0, 10], target=[0, 0, 0],
    projection='orthographic', view_size=3.5,
    legend=[
        {'color': '#ef4444', 'label': 'Arus dinding dalam (keluar)'},
        {'color': '#3b82f6', 'label': 'Arus dinding luar (masuk)'},
        {'color': '#f43f5e', 'label': 'Loop Amperian (a ≤ s ≤ b)'},
        {'color': '#00f5d4', 'label': 'Medan Bφ di dalam inti'}
    ], height=480
)

In [51]:
# Visualisasi 3D: Lilitan Toroida dan Medan Azimutal Terkurung
R_maj = 0.5
r_min = 0.16
N_turns_t = 28
N_eval_t = 1200
th_t = np.linspace(0, 2*np.pi, N_eval_t)
ph_t = N_turns_t*th_t
pts_tor = np.column_stack([
    (R_maj + r_min*np.cos(ph_t))*np.cos(th_t),
    (R_maj + r_min*np.cos(ph_t))*np.sin(th_t),
    r_min*np.sin(ph_t)
])
elements = [_three_tube(pts_tor, '#e76f51', radius=0.012, closed=True, metalness=0.9, roughness=0.15)]
elements.append(_three_torus([0, 0, 0], R_maj, r_min, '#457b9d', opacity=0.22))
angles = np.linspace(0, 2*np.pi, 150)
for ring_radius, color in zip([R_maj - 0.05, R_maj, R_maj + 0.05], ['#06d6a0', '#39ff14', '#06d6a0']):
    ring = np.column_stack([ring_radius*np.cos(angles), ring_radius*np.sin(angles), np.zeros_like(angles)])
    elements.append(_three_tube(ring, color, radius=0.01, closed=True, metalness=0.4, roughness=0.25))
for angle in np.linspace(0, 2*np.pi, 12, endpoint=False):
    start = np.array([R_maj*np.cos(angle), R_maj*np.sin(angle), 0])
    tangent = np.array([-np.sin(angle), np.cos(angle), 0])
    elements.append(_three_arrow(start, start + 0.17*tangent, '#ffd166', radius=0.03, metalness=0.6))
elements.append(_three_floor(-0.6, 2.5, 2.5))
_three_show(
    'Lilitan Toroida dan Garis Medan Magnet Azimutal',
    elements, camera=[1.5, -1.5, 1.3], target=[0, 0, 0],
    legend=[
        {'color': '#e76f51', 'label': f'Lilitan toroida ({N_turns_t} turns)'},
        {'color': '#457b9d', 'label': 'Inti torus transparan'},
        {'color': '#39ff14', 'label': 'Garis medan Bφ terkurung'},
        {'color': '#ffd166', 'label': 'Arah sirkulasi medan'}
    ]
)

---
## 6. Kasus Analitik 5: Pasangan Kumparan Helmholtz (Rekayasa Medan Homogen)

### 6.1 Penurunan Matematis Eksak
Kumparan Helmholtz terdiri atas dua kumparan lingkaran koaksial identik berjari-jari $R$ yang dipisahkan oleh jarak $d$ yang tepat sama dengan jari-jarinya ($d = R$) dan dialiri arus searah $I$.

Misalkan titik tengah kedua kumparan adalah $z = 0$, sehingga kumparan 1 berada di $z = -R/2$ dan kumparan 2 di $z = +R/2$. Berdasarkan superposisi Kasus 2:
$$B_z(z) = \frac{\mu_0 I R^2}{2} \left[ \frac{1}{\left(R^2 + (z + R/2)^2\right)^{3/2}} + \frac{1}{\left(R^2 + (z - R/2)^2\right)^{3/2}} \right]$$

Keunggulan matematis pemilihan $d = R$:
1. **Turunan Pertama Lenyap di Titik Pusat ($z = 0$):**
   $$\left. \frac{\partial B_z}{\partial z} \right|_{z=0} = 0 \quad (\text{akibat simetri genap } B_z(-z) = B_z(z))$$
2. **Turunan Kedua Lenyap di Titik Pusat ($z = 0$):**
   Dengan menghitung turunan kedua terhadap $z$ dan mensubstitusi $d = R$:
   $$\left. \frac{\partial^2 B_z}{\partial z^2} \right|_{z=0} = 0$$

Hasilnya adalah medan magnet yang **sangat seragam** di sekitar $z = 0$, dengan nilai eksak di titik pusat:
$$B_z(0) = \left(\frac{4}{5}\right)^{3/2} \frac{\mu_0 I}{R} \approx 0.7155 \, \frac{\mu_0 I}{R}$$

### 6.2 Penjelasan Fisis & Proyeksi Vektor
Penurunan kelengkungan cekung ke bawah dari kumparan 1 tepat diimbangi oleh penurunan kelengkungan dari kumparan 2 di daerah antara kedua kumparan, menghasilkan profil medan yang mendatar (*flat plateau*) tanpa fluktuasi medan yang berarti.

In [52]:
# Verifikasi Simbolik SymPy: Uji Kehomogenan Medan Kumparan Helmholtz di Pusat (z = 0)
mu0_h, I_h, R_h, z_h = sp.symbols('mu_0 I R z', positive=True, real=True)

# Medan aksial superposisi dua kumparan koaksial berjari-jari R terpisah sejauh d = R
# Kumparan 1 di z = -R/2, Kumparan 2 di z = +R/2
B_helm = (mu0_h * I_h * R_h**2 / 2) * (
    1 / (R_h**2 + (z_h + R_h/2)**2)**sp.Rational(3, 2) +
    1 / (R_h**2 + (z_h - R_h/2)**2)**sp.Rational(3, 2)
)

# Nilai eksak medan di titik pusat simetri z = 0
B_center = sp.simplify(B_helm.subs(z_h, 0))

# Uji Turunan Pertama dan Kedua di z = 0
dB_dz = sp.diff(B_helm, z_h)
d2B_dz2 = sp.diff(dB_dz, z_h)

dB_dz_0 = sp.simplify(dB_dz.subs(z_h, 0))
d2B_dz2_0 = sp.simplify(d2B_dz2.subs(z_h, 0))

print("1. Nilai Eksak Medan di Titik Pusat (z = 0):")
sp.pprint(B_center)

print("\n2. Nilai Turunan Pertama di z = 0 (dB/dz = 0 membuktikan titik stasioner/ekstremum):")
sp.pprint(dB_dz_0)

print("\n3. Nilai Turunan Kedua di z = 0 (d^2B/dz^2 = 0 membuktikan titik belok/kelengkungan datar):")
sp.pprint(d2B_dz2_0)

1. Nilai Eksak Medan di Titik Pusat (z = 0):
8⋅√5⋅I⋅μ₀
─────────
  25⋅R   

2. Nilai Turunan Pertama di z = 0 (dB/dz = 0 membuktikan titik stasioner/ekstremum):
0

3. Nilai Turunan Kedua di z = 0 (d^2B/dz^2 = 0 membuktikan titik belok/kelengkungan datar):
0


In [53]:
# Diagram Proyeksi Vektor 2D: Fluks Medan Kumparan Helmholtz
R_h = 0.5
d_h = R_h
elements = []
for z in [R_h, -R_h]:
    elements.append(_three_sphere([-d_h/2, 0, z], '#ef4444' if z > 0 else '#3b82f6', 0.05))
    elements.append(_three_sphere([d_h/2, 0, z], '#ef4444' if z > 0 else '#3b82f6', 0.05))
elements.append(_three_tube([[-0.8, 0, 0], [0.8, 0, 0]], '#94a3b8', radius=0.008))
for z in [0.0, 0.15, 0.28, -0.15, -0.28]:
    elements.append(_three_arrow([-d_h*0.8, 0, z], [d_h*0.8, 0, z], '#00f5d4', radius=0.025))
elements.append(_three_arrow([0, 0, 0], [0.38, 0, 0], '#39ff14', radius=0.04))
_three_show(
    'Proyeksi Vektor 2D: Fluks Medan Kumparan Helmholtz',
    elements, camera=[0, -10, 0.1], target=[0, 0, 0],
    projection='orthographic', view_size=2.4,
    legend=[
        {'color': '#ef4444', 'label': 'Kumparan 1 (z = +R/2)'},
        {'color': '#3b82f6', 'label': 'Kumparan 2 (z = −R/2)'},
        {'color': '#94a3b8', 'label': 'Sumbu simetri z'},
        {'color': '#00f5d4', 'label': 'Fluks medan homogen'},
        {'color': '#39ff14', 'label': 'Medan sentral B₀'}
    ], height=480
)

In [54]:
# Visualisasi 3D: Pasangan Kumparan Helmholtz dan Zona Homogen Pusat
R_helm = 0.5
d_helm = R_helm
th_h = np.linspace(0, 2*np.pi, 150)
pts_h1 = np.column_stack([R_helm*np.cos(th_h), R_helm*np.sin(th_h), np.full_like(th_h, -d_helm/2)])
pts_h2 = np.column_stack([R_helm*np.cos(th_h), R_helm*np.sin(th_h), np.full_like(th_h, d_helm/2)])
elements = [
    _three_tube(pts_h1, '#d97443', radius=0.032, closed=True, metalness=0.9, roughness=0.16),
    _three_tube(pts_h2, '#d97443', radius=0.032, closed=True, metalness=0.9, roughness=0.16)
]
for angle in [np.pi/4, 3*np.pi/4, 5*np.pi/4, 7*np.pi/4]:
    stand = np.column_stack([
        np.full(30, R_helm*np.cos(angle)), np.full(30, R_helm*np.sin(angle)),
        np.linspace(-d_helm/2, d_helm/2, 30)
    ])
    elements.append(_three_tube(stand, '#4a4e69', radius=0.012, metalness=0.7, roughness=0.3))
for radius in [0.0, 0.12, 0.22]:
    for angle in np.linspace(0, 2*np.pi, 8 if radius > 0 else 1, endpoint=False):
        x, y = radius*np.cos(angle), radius*np.sin(angle)
        elements.append(_three_tube([[x, y, -d_helm/2*1.3], [x, y, d_helm/2*1.3]],
                                    '#00f5d4', radius=0.008, metalness=0.4, roughness=0.25))
elements.append(_three_arrow([0, 0, -0.12], [0, 0, 0.25], '#39ff14', radius=0.035, metalness=0.5))
elements.append(_three_floor(-1.0, 3.0, 3.0))
_three_show(
    'Pasangan Kumparan Helmholtz dan Medan Homogen Pusat',
    elements, camera=[2.0, -2.0, 1.3], target=[0, 0, 0],
    legend=[
        {'color': '#d97443', 'label': 'Kumparan Helmholtz (jarak d = R)'},
        {'color': '#4a4e69', 'label': 'Rangka penyangga'},
        {'color': '#00f5d4', 'label': 'Garis fluks medan homogen'},
        {'color': '#39ff14', 'label': 'Medan pusat B₀'}
    ]
)

---
## 7. Rangkuman Komparatif & Kesimpulan Akhir

| No | Kasus Analitik | Simetri Sistem | Arah Vektor $\mathbf{B}$ | Formula Eksak Medan Magnet | Karakteristik Proyeksi Fisis |
| :---: | :--- | :--- | :--- | :--- | :--- |
| **1** | **Kawat Lurus Tak Hingga** | Silinder translasi & rotasi | Azimutal $\hat{\boldsymbol{\phi}}$ murni | $B = \frac{\mu_0 I}{2\pi s}$ | $B_s = 0, B_z = 0$. Garis gaya konsentris melingkari kawat, meluruh sebagai $1/s$. |
| **2** | **Cincin Melingkar (Sumbu-$z$)** | Rotasi aksial | Aksial $\hat{\mathbf{z}}$ murni | $B_z = \frac{\mu_0 I R^2}{2(R^2+z^2)^{3/2}}$ | Pasangan antipodal saling meniadakan komponen radial ($dB_\perp = 0$), $B \propto 1/z^3$ di jarak jauh. |
| **3** | **Solenoida Berhingga** | Silinder translasi terbatas | Aksial $\hat{\mathbf{z}}$ dominan | $B_z = \frac{\mu_0 n I}{2}(\cos\theta_2 - \cos\theta_1)$ | Di pusat ($L \to \infty$): $\mu_0 n I$; di tepi ujung tepat separuh $\frac{1}{2}\mu_0 n I$. |
| **4** | **Toroida Ideal** | Toroidal tertutup | Azimutal $\hat{\boldsymbol{\phi}}$ terkurung | $B = \frac{\mu_0 N I}{2\pi s}$ ($a \le s \le b$) | Nol di rongga dalam dan luar; medan $100\%$ terkurung di dalam inti donat; $B \propto 1/s$ radial. |
| **5** | **Kumparan Helmholtz** | Aksial ekuatorial ganda | Aksial $\hat{\mathbf{z}}$ seragam | $B_0 = \left(\frac{4}{5}\right)^{3/2}\frac{\mu_0 I}{R}$ | $\frac{\partial B_z}{\partial z} = 0$ dan $\frac{\partial^2 B_z}{\partial z^2} = 0$; menciptakan zona medan seragam $\pm R/4$. |

### Kesimpulan Metodologis:
1. **Analisis Proyeksi Vektor:** Pembatalan komponen medan ortogonal selalu dapat diprediksi melalui pertimbangan simetri grup ($C_\infty, \sigma_h, \sigma_v$) dan pasangan elemen arus antipodal $d\mathbf{l}'$.
2. **Keterkaitan Biot-Savart & Ampère:** Hukum Biot-Savart adalah prinsip diferensial universal (berlaku untuk semua geometri), sedangkan Hukum Ampère menyediakan jalan pintas integral untuk sistem dengan simetri ruang yang tinggi.
3. **Visualisasi Interaktif (Three.js):** Seluruh diagram proyeksi 2D dan visualisasi 3D dirender oleh Three.js di browser. Visualisasi 3D mendukung rotasi dan zoom kamera serta material PBR; pemuatan modul Three.js dilakukan melalui CDN jsDelivr saat notebook menampilkan keluaran visual.